<a href="https://colab.research.google.com/github/Shimba-crypto/shimba-llm/blob/main/shimba_colab.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Shimba LLM Trainer
**One-file LLM trainer — runs on Google Colab CPU/GPU for FREE**

Just click **Runtime → Run all**. No setup needed.

In [ ]:
# @title 1. Install & Download
import os, sys, urllib.request

!pip install numpy -q

# Download shimba.py from GitHub
url = "https://raw.githubusercontent.com/Shimba-crypto/shimba-llm/main/shimba.py"
local = "shimba.py"
if not os.path.exists(local):
    urllib.request.urlretrieve(url, local)
    print(f"Downloaded {local}")

# Mount Google Drive (upload your .txt dataset there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Run smoke test
!python shimba.py test

In [ ]:
# @title Train a tiny model (quick test)
# Creates a small dataset and trains for 50 steps
!python shimba.py train --data shimba.py --out tiny_model.pth --n_embd 64 --n_layer 2 --n_head 2 --block_size 128 --batch_size 8 --max_iters 50 --eval_interval 25

In [ ]:
# @title Train a real model (change these paths)
# Upload your .txt dataset to Google Drive, then set path here
DATASET_PATH = "/content/drive/MyDrive/data.txt"  # ← CHANGE THIS
MODEL_OUT    = "my_model.pth"

# You can also use a dataset from within Colab:
# DATASET_PATH = "training_corpus.txt"

MODEL_DIM    = 256   # embedding size (bigger = smarter)
MODEL_LAYERS = 6     # transformer layers
MODEL_HEADS  = 8     # attention heads
BLOCK_SIZE   = 256   # context window
BATCH_SIZE   = 8     # sequences per batch
MAX_ITERS    = 5000  # total training steps
LEARNING_RATE = 3e-4

import os
if os.path.exists(DATASET_PATH):
    !python shimba.py train --data "$DATASET_PATH" --out "$MODEL_OUT" \
        --n_embd $MODEL_DIM --n_layer $MODEL_LAYERS --n_head $MODEL_HEADS \
        --block_size $BLOCK_SIZE --batch_size $BATCH_SIZE \
        --max_iters $MAX_ITERS --lr $LEARNING_RATE
else:
    print(f"Dataset not found: {DATASET_PATH}")
    print("Upload a .txt file to Google Drive and update DATASET_PATH")

In [ ]:
# @title Chat with trained model
MODEL_PATH = "my_model.pth"  # ← CHANGE to your model

import os
if os.path.exists(MODEL_PATH):
    from shimba import chat_loop, CharTokenizer, GPT
    
    tok = CharTokenizer.load(CharTokenizer.default_path(MODEL_PATH))
    model = GPT.load(MODEL_PATH).to("cuda" if torch.cuda.is_available() else "cpu")
    
    prompt = "User: what is machine learning\nResponse:"
    result = generate(model, tok, prompt, max_new=100, temperature=0.7, top_k=40)
    print(result)
else:
    print("Model not found. Train one first.")

In [ ]:
# @title Benchmark model
MODEL_PATH = "my_model.pth"  # ← CHANGE
import os
if os.path.exists(MODEL_PATH):
    !python shimba.py benchmark --model "$MODEL_PATH"
else:
    print("Model not found.")

In [ ]:
# @title Start API server (runs until stopped)
MODEL_PATH = "my_model.pth"  # ← CHANGE
PORT = 8000

# Run in background so we can test
import threading, json, urllib.request

def run_server():
    !python shimba.py serve --model "$MODEL_PATH" --port $PORT

t = threading.Thread(target=run_server, daemon=True)
t.start()
import time; time.sleep(3)

# Test the server
try:
    req = urllib.request.Request(
        f"http://localhost:{PORT}",
        data=json.dumps({"prompt": "Hello", "max_tokens": 30}).encode(),
        headers={"Content-Type": "application/json"})
    resp = urllib.request.urlopen(req)
    print(json.loads(resp.read()))
except Exception as e:
    print(f"Server test: {e}")

In [ ]:
# @title Download trained model
from google.colab import files
files.download("my_model.pth")
files.download("my_model_tokenizer.json")

---
## Deploy to the cloud

Once trained, deploy your model anywhere:

| Platform | Command |
|----------|---------|
| **Google Cloud Run** | `docker build -t shimba-llm . && gcloud run deploy` |
| **AWS ECS** | `docker build -t shimba-llm . && aws ecs run-task` |
| **Azure ACI** | `docker build -t shimba-llm . && az container create` |
| **Hugging Face** | Connect GitHub repo → HF Spaces auto-builds |

See `deploy.sh` in the repo for full instructions.

**Repo: https://github.com/Shimba-crypto/shimba-llm**